In [1]:
#!pip install -U torch scikit-learn torchvision timm optuna plotly

# Hyperparameter Tuning for: Trustworthy Deep Learning for Chest X-ray Disease Detection
## Stage 2 - Hyperparameter Optimization (DenseNet121)

Builds on the Stage 1 baseline pipeline (same data split, transforms, two-phase transfer
learning) and wraps it in an **Optuna** search over the hyperparameter-sweep knobs the
baseline exposed (optimizer, learning rates, weight decay, scheduler, how many dense blocks
to unfreeze) plus dropout rate and batch size.

Search strategy:
- Each trial trains a **short** version of the two-phase schedule (`--hpo_phase1_epochs`,
  `--hpo_phase2_epochs`, both much smaller than the full run) and is scored on validation
  loss after phase 2.
- A `MedianPruner` stops clearly-unpromising trials early so the search doesn't waste
  compute on bad configurations.
- Once the search finishes, the **best** hyperparameters are used to retrain the model with
  the full epoch budget (same as Stage 1) and evaluate on the held-out test set.

Run on Google Colab / Kaggle (free GPU). Example:
    python densenet121_hpo.py --data_dir /kaggle/input/covid19-radiography-database \
        --output_dir ./runs/densenet121_hpo --n_trials 20

In [2]:
import argparse
import copy
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm

try:
    import timm
except ImportError as e:
    raise ImportError(
        "This script requires `timm`. Install with: pip install timm --break-system-packages"
    ) from e

try:
    import optuna
    from optuna.trial import TrialState
except ImportError as e:
    raise ImportError(
        "This script requires `optuna`. Install with: pip install optuna --break-system-packages"
    ) from e

## Reproducibility

In [3]:
def set_seed(seed: int = 42, deterministic: bool = True):
    """Set the random seed for reproducibility.

    deterministic=True (default) pins cuDNN to deterministic, non-autotuned
    kernels - used for the final full retrain, where exact reproducibility
    matters. During HPO trials we call this with deterministic=False so
    cuDNN's autotuner (benchmark mode) can pick faster convolution
    algorithms - safe here since every trial uses a fixed 224x224 input size.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic

## Data

Identical to the Stage 1 baseline: 70/15/15 stratified split, ImageNet normalization,
train-only augmentation.

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [5]:
def build_transforms(img_size: int = 224):
  train_tf = transforms.Compose(
    [
      transforms.Resize((img_size, img_size)),
      transforms.RandomCrop(img_size, padding=8, padding_mode='reflect'),
      transforms.RandomHorizontalFlip(p=0.5),
      transforms.RandomRotation(degrees=10),
      transforms.ColorJitter(brightness=0.2, contrast=0.2),
      transforms.Grayscale(num_output_channels=3),
      transforms.ToTensor(),
      transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ]
  )

  eval_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
  ])
  return train_tf, eval_tf

In [6]:
class TransformSubset(Dataset):
  """Wraps a Subset of an ImageFolder so train/val/test can each use a different transform."""

  def __init__(self, subset: Subset, transform):
    self.subset = subset
    self.transform = transform

  def __len__(self):
    return len(self.subset)

  def __getitem__(self, idx):
    img, label = self.subset[idx]
    if self.transform is not None:
      img = self.transform(img)
    return img, label

In [7]:
def stratified_split(dataset: ImageFolder, seed: int = 42):
  """70/15/15 stratified split of dataset into train, val, and test subsets."""

  targets = np.array(dataset.targets)
  indices = np.arange(len(dataset))

  train_idx, temp_idx = train_test_split(
    indices, test_size=0.3, stratify=targets, random_state=seed
  )

  val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=targets[temp_idx],
    random_state=seed,
  )

  return train_idx, val_idx, test_idx

In [8]:
def load_base_dataset_and_split(data_dir: str, seed: int):
  """Scan the dataset directory and compute the train/val/test split ONCE.

  Previously this (plus the DataLoader construction) ran inside every Optuna
  trial, which meant re-scanning the whole dataset directory from disk 20+
  times. Now it runs a single time and every trial just reuses the indices.
  """
  def only_images_folder(path):
    p = Path(path)
    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".gif"}
    return p.parent.name.lower() == "images" and p.suffix.lower() in valid_exts
  base_dataset = ImageFolder(root=data_dir, is_valid_file=only_images_folder)

  train_idx, val_idx, test_idx = stratified_split(base_dataset, seed=seed)
  class_names = base_dataset.classes
  train_targets = np.array(base_dataset.targets)[train_idx]
  return base_dataset, train_idx, val_idx, test_idx, class_names, train_targets


def subsample_train_idx(train_idx: np.ndarray, train_targets: np.ndarray, frac: float, seed: int):
  """Stratified subsample of the training indices, used to speed up HPO trials.

  We don't need the full training set to rank hyperparameter configs - a
  smaller stratified slice gives a comparable signal for a fraction of the
  compute. frac=1.0 (or >=1.0) uses the full training set unchanged.
  """
  if frac >= 1.0:
    return train_idx
  sub_idx, _ = train_test_split(
    train_idx, train_size=frac, stratify=train_targets, random_state=seed
  )
  return sub_idx


def make_dataloaders(base_dataset, train_idx, val_idx, test_idx, img_size: int, batch_size: int, num_workers: int = 4):
  """Build train/val/test DataLoaders from already-computed indices (no disk re-scan)."""
  train_tf, eval_tf = build_transforms(img_size)

  train_ds = TransformSubset(Subset(base_dataset, train_idx), train_tf)
  val_ds = TransformSubset(Subset(base_dataset, val_idx), eval_tf)
  test_ds = TransformSubset(Subset(base_dataset, test_idx), eval_tf)

  pin_memory = torch.cuda.is_available()
  persistent = num_workers > 0
  prefetch = 2 if num_workers > 0 else None
  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers,
                             pin_memory=pin_memory, persistent_workers=persistent, prefetch_factor=prefetch)
  val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                           pin_memory=pin_memory, persistent_workers=persistent, prefetch_factor=prefetch)
  test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                            pin_memory=pin_memory, persistent_workers=persistent, prefetch_factor=prefetch)

  datasets = {"train": train_ds, "val": val_ds, "test": test_ds}
  return train_loader, val_loader, test_loader, datasets

In [9]:
def compute_class_weights(train_targets: np.ndarray, num_classes: int) -> torch.Tensor:
  counts = np.bincount(train_targets, minlength=num_classes).astype(np.float32)
  counts[counts == 0] = 1.0  # avoid div-by-zero
  weights = counts.sum() / (num_classes * counts)
  return torch.tensor(weights, dtype=torch.float32)

## Model

Same DenseNet121 backbone as the baseline, but `build_model` now also accepts a tunable
`drop_rate` (dropout before the classifier head) since that's one of the hyperparameters
being searched.

In [10]:
def build_model(num_classes: int = 4, drop_rate: float = 0.0) -> nn.Module:
    model = timm.create_model("densenet121", pretrained=True, num_classes=num_classes, drop_rate=drop_rate)
    return model

In [11]:
def freeze_backbone(model: nn.Module):
    """Phase 1: freeze everything except the final classifier head."""
    for name, param in model.named_parameters():
        if "classifier" in name or "fc" in name:  # timm densenet head is named 'classifier'
            param.requires_grad = True
        else:
            param.requires_grad = False

In [12]:
def unfreeze_final_blocks(model: nn.Module, num_dense_blocks_to_unfreeze: int = 1):
    """Phase 2: unfreeze the classifier + the last N dense blocks for end-to-end fine-tuning."""
    for param in model.parameters():
        param.requires_grad = False
    for name, param in model.named_parameters():
        if "classifier" in name:
            param.requires_grad = True
    # timm densenet121 feature blocks are named features.denseblock1..4 / features.norm5
    unfreeze_names = ["features.norm5"] + [
        f"features.denseblock{4 - i}" for i in range(num_dense_blocks_to_unfreeze)
    ] + [
        f"features.transition{3 - i}" for i in range(num_dense_blocks_to_unfreeze)
    ]
    for name, param in model.named_parameters():
        if any(name.startswith(u) for u in unfreeze_names):
            param.requires_grad = True

In [13]:
def build_optimizer(model, name: str, lr: float, weight_decay: float):
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if name == "adamw":
        return torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "adam":
        return torch.optim.Adam(trainable_params, lr=lr, weight_decay=weight_decay)
    elif name == "sgd":
        return torch.optim.SGD(trainable_params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

In [14]:
def build_scheduler(optimizer, name: str, epochs: int):
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(1, epochs // 3), gamma=0.1)
    elif name == "plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.1, patience=2)
    elif name == "none":
        return None
    raise ValueError(f"Unknown scheduler: {name}")

## Training / evaluation loops

`run_epoch` and `evaluate` are unchanged from the baseline. There are now **two** phase
trainers:
- `train_phase_hpo` — a lean version used *inside* each Optuna trial: no checkpoint/history
  files (trials are short and disposable), and it reports validation loss to the trial each
  epoch so the pruner can cut a bad trial short.
- `train_phase` — the original baseline trainer, used only once at the end to retrain the
  winning configuration with the full epoch budget and write out history/checkpoints.

In [15]:
def run_epoch(model, loader, criterion, optimizer, device, train: bool, scaler=None, desc: str = ""):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    non_blocking = device.type == "cuda"
    use_amp = scaler is not None and scaler.is_enabled()
    context = torch.enable_grad() if train else torch.no_grad()
    progress = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
    with context:
        for images, labels in progress:
            images = images.to(device, non_blocking=non_blocking)
            labels = labels.to(device, non_blocking=non_blocking)
            if train:
                optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if train:
                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += batch_size
            progress.set_postfix(loss=f"{total_loss / total:.4f}", acc=f"{correct / total:.4f}")

    return total_loss / total, correct / total

In [16]:
def train_phase_hpo(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, epochs, patience, trial, step_offset=0, use_amp=True,
):
    """Lean trainer used inside an Optuna trial: early stopping + per-epoch pruning report,
    no disk writes. Returns (model, best_val_loss)."""
    best_val_loss = float("inf")
    epochs_no_improve = 0
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and device.type == "cuda")

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True, scaler=scaler,
            desc=f"trial{trial.number} epoch {epoch}/{epochs} train",
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False, scaler=None,
            desc=f"trial{trial.number} epoch {epoch}/{epochs} val",
        )

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        trial.report(val_loss, step_offset + epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if epochs_no_improve >= patience:
            break

    return model, best_val_loss

In [17]:
def train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    device, epochs, patience, phase_name, output_dir, use_amp=True,
):
    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    history = []
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and device.type == "cuda")

    for epoch in range(1, epochs + 1):
        train_desc = f"{phase_name} epoch {epoch}/{epochs} train"
        val_desc = f"{phase_name} epoch {epoch}/{epochs} val"
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, optimizer, device, train=True, scaler=scaler, desc=train_desc
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, optimizer, device, train=False, scaler=None, desc=val_desc
        )

        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                         "val_loss": val_loss, "val_acc": val_acc})
        print(f"[{phase_name}] epoch {epoch}/{epochs} "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"[{phase_name}] Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
                break

    model.load_state_dict(best_state)
    with open(Path(output_dir) / f"{phase_name}_history.json", "w") as f:
        json.dump(history, f, indent=2)
    return model

In [18]:
@torch.no_grad()
def evaluate(model, loader, class_names, device, output_dir, evaluation_type):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    non_blocking = device.type == "cuda"
    for images, labels in loader:
        images = images.to(device, non_blocking=non_blocking)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds)
        all_probs.extend(probs)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, output_dict=True)
    cm = confusion_matrix(all_labels, all_preds)

    try:
        auc_macro = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
        auc_per_class = roc_auc_score(all_labels, all_probs, multi_class="ovr", average=None)
    except ValueError:
        auc_macro, auc_per_class = None, None

    results = {
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "roc_auc_macro": auc_macro,
        "roc_auc_per_class": auc_per_class.tolist() if auc_per_class is not None else None,
        "class_names": class_names,
    }

    with open(Path(output_dir) / f"{evaluation_type}_test_results.json", "w") as f:
        json.dump(results, f, indent=2)

    print("\n=== Test set performance ===")
    print(f"Test accuracy: {accuracy_score(all_labels, all_preds):.4f}")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    print("Confusion matrix:\n", cm)
    if auc_macro is not None:
        print(f"Macro ROC-AUC: {auc_macro:.4f}")

    # --- Save confusion matrix as CSV ---
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_df.to_csv(Path(output_dir) / f"{evaluation_type}_confusion_matrix.csv")

    # --- Save one-row summary CSV (accuracy, macro/weighted f1 & recall, ROC-AUC) ---
    summary = {
        "accuracy": accuracy_score(all_labels, all_preds),
        "macro_f1": report["macro avg"]["f1-score"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_precision": report["macro avg"]["precision"],
        "weighted_f1": report["weighted avg"]["f1-score"],
        "weighted_recall": report["weighted avg"]["recall"],
        "roc_auc_macro": auc_macro,
    }
    pd.DataFrame([summary]).to_csv(Path(output_dir) / f"{evaluation_type}_summary_metrics.csv", index=False)

    return results

## Config

Two epoch budgets are kept separate on purpose:
- `--hpo_phase1_epochs` / `--hpo_phase2_epochs` / `--hpo_patience` — short, used **inside**
  every Optuna trial, so the search itself stays tractable.
- `--phase1_epochs` / `--phase2_epochs` / `--patience` — the full baseline budget, used only
  **once** at the end to retrain the winning configuration.

In [19]:
parser = argparse.ArgumentParser(description="DenseNet121 hyperparameter tuning - Stage 2")
parser.add_argument("--data_dir", type=str,
                    default=r"/kaggle/input/datasets/tawsifurrahman/covid19-radiography-database/COVID-19_Radiography_Dataset",
                    help="Path to dataset root with one subfolder per class")
parser.add_argument("--output_dir", type=str, default="./runs/densenet121_hpo")
parser.add_argument("--img_size", type=int, default=224)
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--num_workers", type=int, default=4)
parser.add_argument("--device", type=str, default="auto", choices=["auto", "cuda", "cpu"],
                     help="'auto' picks CUDA if available, else CPU. Force 'cpu' to sanity-check "
                          "the pipeline locally before running full training on a GPU box.")
parser.add_argument("--disable_amp", action="store_true",
                     help="Disable mixed-precision (AMP) training. AMP is ON by default and "
                          "roughly halves per-epoch GPU time with no accuracy cost.")

# --- HPO search budget (short per trial, so the search stays tractable) ---
parser.add_argument("--n_trials", type=int, default=20)
parser.add_argument("--hpo_phase1_epochs", type=int, default=5)
parser.add_argument("--hpo_phase2_epochs", type=int, default=8)
parser.add_argument("--hpo_patience", type=int, default=3)
parser.add_argument("--hpo_train_frac", type=float, default=0.3,
                     help="Stratified fraction of the training set used INSIDE each HPO trial. "
                          "1.0 = full training set. Lower = faster search, used only for ranking "
                          "configs; the final retrain below always uses the full training set.")
parser.add_argument("--hpo_n_startup_trials", type=int, default=3,
                     help="Trials that always run to completion before the pruner can cut a trial short.")
parser.add_argument("--hpo_n_warmup_steps", type=int, default=1,
                     help="Epochs each trial gets before it becomes eligible for pruning.")
parser.add_argument("--study_name", type=str, default="densenet121_covid_hpo")
parser.add_argument("--storage", type=str, default=None,
                     help="Optuna storage URL, e.g. sqlite:///./runs/densenet121_hpo/study.db, "
                          "to persist/resume the study across sessions. None keeps it in-memory.")

# --- Final retrain budget, once the best hyperparameters are found (same as Stage 1 baseline) ---
parser.add_argument("--phase1_epochs", type=int, default=15)
parser.add_argument("--phase2_epochs", type=int, default=40)
parser.add_argument("--patience", type=int, default=5)

_StoreAction(option_strings=['--patience'], dest='patience', nargs=None, const=None, default=5, type=<class 'int'>, choices=None, required=False, help=None, metavar=None)

In [20]:
args, _ = parser.parse_known_args()

Path(args.output_dir).mkdir(parents=True, exist_ok=True)
set_seed(args.seed)

In [21]:
if args.device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("--device cuda was requested but no CUDA GPU is available.")
if args.device == "auto":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
    device = torch.device(args.device)

print(f"Using device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(device)}")
    print(f"  CUDA version: {torch.version.cuda}")
    total_mem_gb = torch.cuda.get_device_properties(device).total_memory / (1024 ** 3)
    print(f"  GPU memory: {total_mem_gb:.1f} GB")
else:
    print("  Running on CPU - the search below will be very slow; "
          "use --device cpu only for a quick smoke test with --n_trials 1.")

Using device: cuda
  GPU: Tesla T4
  CUDA version: 12.8
  GPU memory: 14.6 GB


In [22]:
# Scan the dataset directory and compute the stratified split ONCE, up front.
# Every HPO trial and the final retrain below reuse these same indices instead
# of re-scanning the dataset directory from disk each time.
base_dataset, train_idx, val_idx, test_idx, class_names, train_targets = load_base_dataset_and_split(
    args.data_dir, args.seed
)
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")
print(f"Train/Val/Test sizes: {len(train_idx)}/{len(val_idx)}/{len(test_idx)}")
if args.hpo_train_frac < 1.0:
    print(f"HPO trials will use a {args.hpo_train_frac:.0%} stratified subsample of the training set "
          f"(~{int(len(train_idx) * args.hpo_train_frac)} images) for speed.")

Classes (4): ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
Train/Val/Test sizes: 14815/3175/3175
HPO trials will use a 30% stratified subsample of the training set (~4444 images) for speed.


## Search space & objective

Search space (mirrors the baseline's "Hyperparameter-sweep knobs" plus dropout and batch
size):

| Hyperparameter    | Range / choices                     |
|--------------------|--------------------------------------|
| `optimizer`        | `adamw`, `adam`, `sgd`               |
| `phase1_lr`         | log-uniform 1e-4 – 1e-2              |
| `phase2_lr`         | log-uniform 1e-6 – 1e-4              |
| `weight_decay`      | log-uniform 1e-6 – 1e-2              |
| `scheduler`         | `cosine`, `step`, `plateau`          |
| `unfreeze_blocks`   | 1 – 3 dense blocks                   |
| `drop_rate`         | 0.0 – 0.5                            |
| `batch_size`        | 32, 64                                |

Each trial reruns the baseline's two-phase schedule (frozen head, then partial fine-tune)
at the short `--hpo_*` epoch budget and is scored on the best validation loss reached in
phase 2 (lower is better).

In [23]:
def objective(trial: "optuna.trial.Trial") -> float:
    optimizer_name = trial.suggest_categorical("optimizer", ["adamw", "adam", "sgd"])
    phase1_lr = trial.suggest_float("phase1_lr", 1e-4, 1e-2, log=True)
    phase2_lr = trial.suggest_float("phase2_lr", 1e-6, 1e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    scheduler_name = trial.suggest_categorical("scheduler", ["cosine", "step", "plateau"])
    unfreeze_blocks = trial.suggest_int("unfreeze_blocks", 1, 3)
    drop_rate = trial.suggest_float("drop_rate", 0.0, 0.5)
    batch_size = trial.suggest_categorical("batch_size", [32, 64])

    # deterministic=False lets cuDNN autotune (benchmark mode) - safe here since
    # every trial uses a fixed 224x224 input size, and meaningfully faster than
    # the fully-deterministic settings reserved for the final retrain.
    set_seed(args.seed, deterministic=False)

    trial_train_idx = subsample_train_idx(train_idx, train_targets, args.hpo_train_frac, args.seed)
    trial_train_loader, trial_val_loader, _, _ = make_dataloaders(
        base_dataset, trial_train_idx, val_idx, test_idx, args.img_size, batch_size, args.num_workers
    )
    trial_train_targets = np.array(base_dataset.targets)[trial_train_idx]
    trial_class_weights = compute_class_weights(trial_train_targets, num_classes).to(device)
    trial_criterion = nn.CrossEntropyLoss(weight=trial_class_weights)

    model = build_model(num_classes=num_classes, drop_rate=drop_rate).to(device)
    use_amp = not args.disable_amp

    # Phase 1: frozen backbone, train head only
    freeze_backbone(model)
    opt1 = build_optimizer(model, optimizer_name, phase1_lr, weight_decay)
    sched1 = build_scheduler(opt1, scheduler_name, args.hpo_phase1_epochs)
    model, _ = train_phase_hpo(
        model, trial_train_loader, trial_val_loader, trial_criterion, opt1, sched1,
        device, args.hpo_phase1_epochs, args.hpo_patience, trial, step_offset=0, use_amp=use_amp,
    )

    # Phase 2: fine-tune final dense blocks
    unfreeze_final_blocks(model, unfreeze_blocks)
    opt2 = build_optimizer(model, optimizer_name, phase2_lr, weight_decay)
    sched2 = build_scheduler(opt2, scheduler_name, args.hpo_phase2_epochs)
    model, best_val_loss = train_phase_hpo(
        model, trial_train_loader, trial_val_loader, trial_criterion, opt2, sched2,
        device, args.hpo_phase2_epochs, args.hpo_patience, trial, step_offset=args.hpo_phase1_epochs, use_amp=use_amp,
    )

    return best_val_loss

## Run the study

`TPESampler` (seeded) proposes configurations; `MedianPruner` cuts a trial short once its
validation loss trajectory falls behind the median of previous trials at the same step.

In [24]:
sampler = optuna.samplers.TPESampler(seed=args.seed)
pruner = optuna.pruners.MedianPruner(n_startup_trials=args.hpo_n_startup_trials, n_warmup_steps=args.hpo_n_warmup_steps)

study = optuna.create_study(
    study_name=args.study_name,
    direction="minimize",
    sampler=sampler,
    pruner=pruner,
    storage=args.storage,
    load_if_exists=True,
)

study.optimize(objective, n_trials=args.n_trials, gc_after_trial=True)

pruned = [t for t in study.trials if t.state == TrialState.PRUNED]
completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
print(f"\nFinished: {len(study.trials)} trials ({len(pruned)} pruned, {len(completed)} completed)")

best_trial = study.best_trial
print("\nBest trial:")
print(f"  Value (val_loss): {best_trial.value:.4f}")
print("  Params:")
for k, v in best_trial.params.items():
    print(f"    {k}: {v}")

[I 2026-09-02 12:02:14,779] A new study created in memory with name: densenet121_covid_hpo


model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]

/tmp/ipykernel_23/1908890044.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and device.type == "cuda")
[I 2026-09-02 12:07:57,212] Trial 0 finished with value: 0.5074308387688764 and parameters: {'optimizer': 'adam', 'phase1_lr': 0.0015751320499779737, 'phase2_lr': 2.05133826308745e-06, 'weight_decay': 4.207053950287936e-06, 'scheduler': 'step', 'unfreeze_blocks': 3, 'drop_rate': 0.010292247147901223, 'batch_size': 32}. Best is trial 0 with value: 0.5074308387688764.
/tmp/ipykernel_23/1908890044.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and device.type == "cuda")
[I 2026-09-02 12:13:02,007] Trial 1 finished with value: 0.7305165375687006 and parameters: {'optimizer': 'adamw', 'phase1_lr': 0.00040596116104843


Finished: 20 trials (8 pruned, 12 completed)

Best trial:
  Value (val_loss): 0.2354
  Params:
    optimizer: adam
    phase1_lr: 0.004119839624605189
    phase2_lr: 6.197015748809147e-05
    weight_decay: 1.8707420137660263e-05
    scheduler: plateau
    unfreeze_blocks: 3
    drop_rate: 0.4303652916281717
    batch_size: 64


In [25]:
best_params_path = Path(args.output_dir) / "best_hyperparameters.json"
with open(best_params_path, "w") as f:
    json.dump({"value": best_trial.value, "params": best_trial.params}, f, indent=2)
print(f"Saved best hyperparameters to {best_params_path}")

trials_df = study.trials_dataframe()
trials_csv_path = Path(args.output_dir) / "optuna_trials.csv"
trials_df.to_csv(trials_csv_path, index=False)
print(f"Saved all trial results to {trials_csv_path}")
trials_df.sort_values("value").head(10)

Saved best hyperparameters to runs/densenet121_hpo/best_hyperparameters.json
Saved all trial results to runs/densenet121_hpo/optuna_trials.csv


,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_drop_rate,params_optimizer,params_phase1_lr,params_phase2_lr,params_scheduler,params_unfreeze_blocks,params_weight_decay,state
9,9,0.235437,2026-09-02 12:43:11.495400,2026-09-02 12:48:58.225995,0 days 00:05:46.730595,64,0.430365,adam,0.004120,0.000062,plateau,3,0.000019,COMPLETE
16,16,0.236827,2026-09-02 13:09:30.026270,2026-09-02 13:15:00.458848,0 days 00:05:30.432578,64,0.385537,adam,0.005539,0.000063,plateau,2,0.000350,COMPLETE
14,14,0.262724,2026-09-02 13:02:39.909315,2026-09-02 13:08:24.259060,0 days 00:05:44.349745,64,0.234636,adam,0.005932,0.000024,plateau,3,0.000026,COMPLETE
12,12,0.277629,2026-09-02 12:55:47.120913,2026-09-02 13:01:33.427881,0 days 00:05:46.306968,64,0.244602,adamw,0.009954,0.000032,plateau,3,0.000070,COMPLETE
11,11,0.279774,2026-09-02 12:50:04.023522,2026-09-02 12:55:46.840687,0 days 00:05:42.817165,64,0.280419,adamw,0.009791,0.000027,plateau,3,0.000040,COMPLETE
8,8,0.356057,2026-09-02 12:37:18.057562,2026-09-02 12:43:11.215286,0 days 00:05:53.157724,32,0.316702,adamw,0.003244,0.000003,plateau,3,0.000002,COMPLETE
7,7,0.408187,2026-09-02 12:31:55.365302,2026-09-02 12:37:17.782274,0 days 00:05:22.416972,64,0.318205,adam,0.003483,0.000010,cosine,1,0.000123,COMPLETE
6,6,0.467188,2026-09-02 12:26:25.495510,2026-09-02 12:31:55.090509,0 days 00:05:29.594999,32,0.443606,sgd,0.001764,0.000005,plateau,2,0.000002,COMPLETE
4,4,0.502025,2026-09-02 12:19:56.733760,2026-09-02 12:25:19.405062,0 days 00:05:22.671302,32,0.414369,adamw,0.006978,0.000002,plateau,1,0.000006,COMPLETE
0,0,0.507431,2026-09-02 12:02:14.780526,2026-09-02 12:07:57.212633,0 days 00:05:42.432107,32,0.010292,adam,0.001575,0.000002,step,3,0.000004,COMPLETE


## Visualize the search (optional)

Requires `plotly`. Falls back gracefully if it isn't installed.

In [26]:
try:
    from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice
    plot_optimization_history(study).show()
    plot_param_importances(study).show()
    plot_slice(study).show()
except Exception as e:
    print(f"Skipping interactive plots ({e}). Install `plotly` for visualizations, "
          "or use optuna.visualization.matplotlib instead.")

## Retrain with the best hyperparameters

Same two-phase procedure as the Stage 1 baseline, but using `best_trial.params` and the
full (`--phase1_epochs` / `--phase2_epochs`) epoch budget, with early stopping and full
history/checkpoint logging via the original `train_phase`.

In [27]:
best = best_trial.params

train_loader, val_loader, test_loader, datasets = make_dataloaders(
    base_dataset, train_idx, val_idx, test_idx, args.img_size, best["batch_size"], args.num_workers
)
print(f"Train/Val/Test sizes: {len(train_loader.dataset)}/{len(val_loader.dataset)}/{len(test_loader.dataset)}")

class_weights = compute_class_weights(train_targets, num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# deterministic=True here (unlike HPO trials) since this is the one run whose
# result matters and should be exactly reproducible.
set_seed(args.seed, deterministic=True)
model = build_model(num_classes=num_classes, drop_rate=best["drop_rate"]).to(device)

Train/Val/Test sizes: 14815/3175/3175


In [28]:
freeze_backbone(model)
opt1 = build_optimizer(model, best["optimizer"], best["phase1_lr"], best["weight_decay"])
sched1 = build_scheduler(opt1, best["scheduler"], args.phase1_epochs)
model = train_phase(
    model, train_loader, val_loader, criterion, opt1, sched1,
    device, args.phase1_epochs, args.patience, "phase1_frozen_best", args.output_dir,
    use_amp=not args.disable_amp,
)

/tmp/ipykernel_23/3900982675.py:9: FutureWarning:

`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.



[phase1_frozen_best] epoch 1/15 train_loss=0.6985 train_acc=0.6834 val_loss=0.4813 val_acc=0.7991


[phase1_frozen_best] epoch 2/15 train_loss=0.5919 train_acc=0.7373 val_loss=0.4398 val_acc=0.8129


[phase1_frozen_best] epoch 3/15 train_loss=0.6000 train_acc=0.7371 val_loss=0.3972 val_acc=0.8258


[phase1_frozen_best] epoch 4/15 train_loss=0.5797 train_acc=0.7443 val_loss=0.3886 val_acc=0.8208


...

[phase1_frozen_best] epoch 11/15 train_loss=0.5270 train_acc=0.7625 val_loss=0.3654 val_acc=0.8444


[phase1_frozen_best] epoch 12/15 train_loss=0.5043 train_acc=0.7704 val_loss=0.3790 val_acc=0.8435


[phase1_frozen_best] epoch 13/15 train_loss=0.5144 train_acc=0.7713 val_loss=0.3891 val_acc=0.8413


[phase1_frozen_best] epoch 14/15 train_loss=0.5170 train_acc=0.7658 val_loss=0.3694 val_acc=0.8450


[phase1_frozen_best] epoch 15/15 train_loss=0.5052 train_acc=0.7711 val_loss=0.3667 val_acc=0.8454


In [29]:
evaluate(model, test_loader, class_names, device, args.output_dir, "frozen_best")


=== Test set performance ===
Test accuracy: 0.8469
                 precision    recall  f1-score   support

          COVID     0.7666    0.8118    0.7885       542
   Lung_Opacity     0.8563    0.7860    0.8197       902
         Normal     0.8727    0.8829    0.8778      1529
Viral Pneumonia     0.8407    0.9406    0.8879       202

       accuracy                         0.8469      3175
      macro avg     0.8340    0.8553    0.8434      3175
   weighted avg     0.8479    0.8469    0.8467      3175

Confusion matrix:
 [[ 440   38   60    4]
 [  65  709  126    2]
 [  69   80 1350   30]
 [   0    1   11  190]]
Macro ROC-AUC: 0.9664


{'classification_report': {'COVID': {'precision': 0.7665505226480837,
   'recall': 0.8118081180811808,
   'f1-score': 0.7885304659498208,
   'support': 542.0},
  'Lung_Opacity': {'precision': 0.856280193236715,
   'recall': 0.7860310421286031,
   'f1-score': 0.8196531791907514,
   'support': 902.0},
  'Normal': {'precision': 0.8726567550096962,
   'recall': 0.8829300196206671,
   'f1-score': 0.8777633289986996,
   'support': 1529.0},
  'Viral Pneumonia': {'precision': 0.8407079646017699,
   'recall': 0.9405940594059405,
   'f1-score': 0.8878504672897196,
   'support': 202.0},
  'accuracy': 0.8469291338582677,
  'macro avg': {'precision': 0.8340488588740662,
   'recall': 0.8553408098090979,
   'f1-score': 0.8434493603572478,
   'support': 3175.0},
  'weighted avg': {'precision': 0.8478583637272948,
   'recall': 0.8469291338582677,
   'f1-score': 0.8466634975138253,
   'support': 3175.0}},
 'confusion_matrix': [[440, 38, 60, 4],
  [65, 709, 126, 2],
  [69, 80, 1350, 30],
  [0, 1, 11, 190

In [30]:
unfreeze_final_blocks(model, best["unfreeze_blocks"])
opt2 = build_optimizer(model, best["optimizer"], best["phase2_lr"], best["weight_decay"])
sched2 = build_scheduler(opt2, best["scheduler"], args.phase2_epochs)
model = train_phase(
    model, train_loader, val_loader, criterion, opt2, sched2,
    device, args.phase2_epochs, args.patience, "phase2_finetune_best", args.output_dir,
    use_amp=not args.disable_amp,
)

/tmp/ipykernel_23/3900982675.py:9: FutureWarning:

`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.



[phase2_finetune_best] epoch 1/40 train_loss=0.3803 train_acc=0.8240 val_loss=0.2310 val_acc=0.9093


[phase2_finetune_best] epoch 2/40 train_loss=0.2546 train_acc=0.8825 val_loss=0.1834 val_acc=0.9206


[phase2_finetune_best] epoch 3/40 train_loss=0.1988 train_acc=0.9073 val_loss=0.1881 val_acc=0.9269


[phase2_finetune_best] epoch 4/40 train_loss=0.1814 train_acc=0.9144 val_loss=0.1582 val_acc=0.9313


...

[phase2_finetune_best] epoch 14/40 train_loss=0.0921 train_acc=0.9563 val_loss=0.1355 val_acc=0.9449


[phase2_finetune_best] epoch 15/40 train_loss=0.0878 train_acc=0.9560 val_loss=0.1389 val_acc=0.9455


[phase2_finetune_best] epoch 16/40 train_loss=0.0873 train_acc=0.9571 val_loss=0.1342 val_acc=0.9468


[phase2_finetune_best] epoch 17/40 train_loss=0.0820 train_acc=0.9575 val_loss=0.1397 val_acc=0.9471


[phase2_finetune_best] epoch 18/40 train_loss=0.0885 train_acc=0.9550 val_loss=0.1328 val_acc=0.9461
[phase2_finetune_best] Early stopping at epoch 18 (no improvement for 5 epochs).


In [31]:
ckpt_path = Path(args.output_dir) / "densenet121_best_hpo.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "best_hyperparameters": best,
}, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")

Saved checkpoint to runs/densenet121_hpo/densenet121_best_hpo.pt


In [32]:
# ---- Evaluate the HPO-tuned model on the held-out test set ----
evaluate(model, test_loader, class_names, device, args.output_dir, "unfreeze_finetune_best")


=== Test set performance ===
Test accuracy: 0.9483
                 precision    recall  f1-score   support

          COVID     0.9742    0.9760    0.9751       542
   Lung_Opacity     0.9480    0.9091    0.9281       902
         Normal     0.9399    0.9614    0.9505      1529
Viral Pneumonia     0.9458    0.9505    0.9481       202

       accuracy                         0.9483      3175
      macro avg     0.9520    0.9493    0.9505      3175
   weighted avg     0.9484    0.9483    0.9482      3175

Confusion matrix:
 [[ 529    5    6    2]
 [   3  820   79    0]
 [  10   40 1470    9]
 [   1    0    9  192]]
Macro ROC-AUC: 0.9928


{'classification_report': {'COVID': {'precision': 0.9742173112338858,
   'recall': 0.9760147601476015,
   'f1-score': 0.9751152073732718,
   'support': 542.0},
  'Lung_Opacity': {'precision': 0.9479768786127167,
   'recall': 0.9090909090909091,
   'f1-score': 0.9281267685342388,
   'support': 902.0},
  'Normal': {'precision': 0.9398976982097187,
   'recall': 0.961412688031393,
   'f1-score': 0.950533462657614,
   'support': 1529.0},
  'Viral Pneumonia': {'precision': 0.9458128078817734,
   'recall': 0.9504950495049505,
   'f1-score': 0.9481481481481482,
   'support': 202.0},
  'accuracy': 0.9483464566929134,
  'macro avg': {'precision': 0.9519761739845236,
   'recall': 0.9492533516937135,
   'f1-score': 0.9504808966783183,
   'support': 3175.0},
  'weighted avg': {'precision': 0.9484279354180204,
   'recall': 0.9483464566929134,
   'f1-score': 0.948212402501926,
   'support': 3175.0}},
 'confusion_matrix': [[529, 5, 6, 2],
  [3, 820, 79, 0],
  [10, 40, 1470, 9],
  [1, 0, 9, 192]],
 'ro